# tcOCR — chạy thử trên Google Colab

OCR tài chính chứng khoán 3 lớp (pluggable) + Gradio UI.

**Trước khi chạy:** Runtime → Change runtime type → chọn **GPU (T4)**.

⚠️ Link Gradio `share=True` là công khai — chỉ test tài liệu **báo cáo tài chính công khai**, KHÔNG dùng data thật của khách.

## 1. Lấy code

In [ ]:
# Cách A: clone từ GitHub (đổi URL/nhánh cho đúng repo của bạn)
import os
if not os.path.exists('tcocr/tcocr'):
    !git clone -b claude/ocr-finance-99-percent-b5o5rh https://github.com/vnkiddev/tcocr.git
%cd tcocr

# Cách B: nếu đã upload thư mục tcOCR lên Colab thì bỏ cell trên, chỉ %cd vào đó.

## 2. Cài thư viện (~5-8 phút lần đầu)

In [ ]:
!pip install -q numpy opencv-python-headless pillow PyMuPDF pdfplumber gradio
!pip install -q paddlepaddle-gpu paddleocr   # code hỗ trợ cả PaddleOCR 2.x lẫn 3.x
!pip install -q vietocr
!pip install -q 'transformers>=4.49' accelerate qwen-vl-utils bitsandbytes sentencepiece
print('Cài xong.')
print('QUAN TRONG: Sau khi cai, Runtime -> Restart session MOT LAN roi chay tiep tu cell 1.')
print('Viec nay tranh loi "PDX has already been initialized" cua PaddleOCR 3.x.')

## 3. Khởi động Gradio UI

Mặc định: OCR=Paddle, escalation=null (tắt VLM cho nhẹ), correction=protonx.
Trên UI đổi backend để so sánh. Bật `local_vlm` khi muốn test Qwen2.5-VL (tốn GPU hơn).

> Nếu vẫn gặp `PDX has already been initialized`: **Runtime → Restart session** rồi chạy lại từ cell 1 (đừng chạy OCR 2 lần trước khi restart).

In [ ]:
from app import build_ui
build_ui().launch(share=True)  # share=True -> link công khai test được ngay

## 4. (Tuỳ chọn) Chạy benchmark bằng code thay vì UI

In [ ]:
from tcocr.config import PipelineConfig
from tcocr.pipeline import OCRPipeline
from tcocr.benchmark.runner import run_pair

cfg = PipelineConfig(ocr_backend='paddle', escalation_backend='null', correction_backend='protonx')
pipe = OCRPipeline(cfg)
report = run_pair(pipe, 'scan.pdf', 'goc.pdf')   # đổi đường dẫn 2 file của bạn
print(report.to_markdown())

# Đổi ocr_backend='vietocr' rồi chạy lại để so Paddle vs VietOCR.